# Monash FAQ: QLoRA versus retrieval
Research pilot with fixed splits, validation-only gate calibration, four generative configurations, and two post-processing baselines.
See the repository protocol for access-to-information differences, the small evaluation size, and reviewer limitations.
Run with Kaggle Internet enabled and a T4 GPU. This notebook retrains the adapter before comparison.

# Monash FAQ Dataset and QLoRA Fine-Tuning

This Notebook contains 58 English question-answer summaries from 10 official Monash public pages, checked on 2026-10-08. Answers are concise paraphrases with source URLs, rather than complete page exports. Topics include applications, VTAC preferences, scholarships, accommodation, library borrowing, postgraduate study, Monash Online and Monash Scholars.

This is a small pilot dataset for testing the workflow, not evidence of production-level factual accuracy. The training split has 43 records, validation has 5, and test has 10.

Kaggle: enable Internet, select GPU T4 x2, and run all cells. The Kaggle edition uses only the first GPU to avoid copying the quantized model through DataParallel. Colab: select a T4 GPU. Data checks do not require a GPU; model training requires NVIDIA CUDA.

The workflow checks data, loads the model, records baseline answers, trains with assistant-only loss, selects the best validation checkpoint, compares test answers, exports an adapter, and reloads it for inference. English training does not establish quality in other languages.

Model: [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct). Method: [PEFT quantized fine-tuning](https://huggingface.co/docs/peft/developer_guides/quantization).


## 1. Install pinned dependencies

Keep the platform-provided CUDA PyTorch installation. Local Jupyter users need PyTorch compatible with their NVIDIA runtime. If an import reports a version conflict after installation, restart the kernel and continue from the data section.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.18.1', 'accelerate==1.12.0',
    'bitsandbytes==0.49.2', 'requests==2.32.5', 'beautifulsoup4==4.14.3'])


## 2. Load the embedded dataset

Questions, answers and messages are English. Each record retains its audience, official source URL and observation date. Training: 43 records; validation: 5; test: 10. Each topic contributes one held-out test question. Validation selects the checkpoint; test references are never added to inference prompts.


In [ ]:
from pathlib import Path
import json, csv, random, hashlib, math, os, platform
from collections import Counter
from IPython.display import display
ROOT = (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()) / 'monash_finetune_run'
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)


In [ ]:
records = [{'id': 'international_application_01', 'group_id': 'international_application_01', 'topic': 'international_application', 'audience': 'international applicants', 'question': 'How long might an international application assessment take?', 'answer': 'Complete current-intake applications usually take one to four weeks; busy periods can take longer.', 'source_url': 'https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How long might an international application assessment take?'}, {'role': 'assistant', 'content': 'Complete current-intake applications usually take one to four weeks; busy periods can take longer.\nSource: https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq'}]}, {'id': 'international_application_02', 'group_id': 'international_application_02', 'topic': 'international_application', 'audience': 'international applicants', 'question': 'What does a conditional Monash admission offer require?', 'answer': 'Additional evidence is needed before a full offer, such as final transcripts, graduation evidence or English results.', 'source_url': 'https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'validation', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What does a conditional Monash admission offer require?'}, {'role': 'assistant', 'content': 'Additional evidence is needed before a full offer, such as final transcripts, graduation evidence or English results.\nSource: https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq'}]}, {'id': 'international_application_03', 'group_id': 'international_application_03', 'topic': 'international_application', 'audience': 'international applicants', 'question': 'How can international applicants submit extra application documents?', 'answer': 'Use my.app, your agent, or email mu.documents@monash.edu. Admissions will request missing evidence if needed.', 'source_url': 'https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How can international applicants submit extra application documents?'}, {'role': 'assistant', 'content': 'Use my.app, your agent, or email mu.documents@monash.edu. Admissions will request missing evidence if needed.\nSource: https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq'}]}, {'id': 'international_application_04', 'group_id': 'international_application_04', 'topic': 'international_application', 'audience': 'international applicants', 'question': 'How should international applicants change their selected course?', 'answer': 'Send the request to mu.documents@monash.edu.', 'source_url': 'https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How should international applicants change their selected course?'}, {'role': 'assistant', 'content': 'Send the request to mu.documents@monash.edu.\nSource: https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq'}]}, {'id': 'international_application_05', 'group_id': 'international_application_05', 'topic': 'international_application', 'audience': 'international applicants', 'question': 'Where is the required offer acceptance deposit specified?', 'answer': 'Use the amount and acceptance deadline in your individual offer letter.', 'source_url': 'https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Where is the required offer acceptance deposit specified?'}, {'role': 'assistant', 'content': 'Use the amount and acceptance deadline in your individual offer letter.\nSource: https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq'}]}, {'id': 'international_application_06', 'group_id': 'international_application_06', 'topic': 'international_application', 'audience': 'international applicants', 'question': 'Who assists international students with unit enrolment difficulties?', 'answer': 'Contact Monash Connect by online enquiry or call +61 3 9902 6011.', 'source_url': 'https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Who assists international students with unit enrolment difficulties?'}, {'role': 'assistant', 'content': 'Contact Monash Connect by online enquiry or call +61 3 9902 6011.\nSource: https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq'}]}, {'id': 'vtac_preferences_01', 'group_id': 'vtac_preferences_01', 'topic': 'vtac_preferences', 'audience': 'VTAC applicants', 'question': 'What changes are possible during VTAC preference windows?', 'answer': 'You can add, remove or rearrange courses during permitted windows; some courses have additional restrictions.', 'source_url': 'https://www.monash.edu/change-of-preference/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What changes are possible during VTAC preference windows?'}, {'role': 'assistant', 'content': 'You can add, remove or rearrange courses during permitted windows; some courses have additional restrictions.\nSource: https://www.monash.edu/change-of-preference/faqs'}]}, {'id': 'vtac_preferences_02', 'group_id': 'vtac_preferences_02', 'topic': 'vtac_preferences', 'audience': 'VTAC applicants', 'question': 'Why does my VTAC course ordering affect offers?', 'answer': 'The offer is for your highest eligible preference, so rank courses according to what you want to study.', 'source_url': 'https://www.monash.edu/change-of-preference/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Why does my VTAC course ordering affect offers?'}, {'role': 'assistant', 'content': 'The offer is for your highest eligible preference, so rank courses according to what you want to study.\nSource: https://www.monash.edu/change-of-preference/faqs'}]}, {'id': 'vtac_preferences_03', 'group_id': 'vtac_preferences_03', 'topic': 'vtac_preferences', 'audience': 'VTAC applicants', 'question': 'How do I edit my VTAC course list?', 'answer': 'Sign into VTAC with your account details and open the course preference list.', 'source_url': 'https://www.monash.edu/change-of-preference/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How do I edit my VTAC course list?'}, {'role': 'assistant', 'content': 'Sign into VTAC with your account details and open the course preference list.\nSource: https://www.monash.edu/change-of-preference/faqs'}]}, {'id': 'vtac_preferences_04', 'group_id': 'vtac_preferences_04', 'topic': 'vtac_preferences', 'audience': 'VTAC applicants', 'question': 'Is there a charge for changing VTAC preferences?', 'answer': 'Changes are free and can be repeated before the relevant deadline.', 'source_url': 'https://www.monash.edu/change-of-preference/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Is there a charge for changing VTAC preferences?'}, {'role': 'assistant', 'content': 'Changes are free and can be repeated before the relevant deadline.\nSource: https://www.monash.edu/change-of-preference/faqs'}]}, {'id': 'vtac_preferences_05', 'group_id': 'vtac_preferences_05', 'topic': 'vtac_preferences', 'audience': 'VTAC applicants', 'question': 'Can a scholarship alone secure a Monash course place?', 'answer': 'No. A scholarship remains subject to receiving a course offer.', 'source_url': 'https://www.monash.edu/change-of-preference/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can a scholarship alone secure a Monash course place?'}, {'role': 'assistant', 'content': 'No. A scholarship remains subject to receiving a course offer.\nSource: https://www.monash.edu/change-of-preference/faqs'}]}, {'id': 'vtac_preferences_06', 'group_id': 'vtac_preferences_06', 'topic': 'vtac_preferences', 'audience': 'VTAC applicants', 'question': 'How do VTAC merit scholarship assessments work at Monash?', 'answer': 'Listing a Monash course in your preferences enables automatic assessment based on your ATAR.', 'source_url': 'https://www.monash.edu/change-of-preference/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How do VTAC merit scholarship assessments work at Monash?'}, {'role': 'assistant', 'content': 'Listing a Monash course in your preferences enables automatic assessment based on your ATAR.\nSource: https://www.monash.edu/change-of-preference/faqs'}]}, {'id': 'online_study_01', 'group_id': 'online_study_01', 'topic': 'online_study', 'audience': 'Monash Online students', 'question': 'Do Monash Online courses require campus classes?', 'answer': 'They are delivered entirely online, without required campus attendance.', 'source_url': 'https://online.monash.edu/faqs/', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Do Monash Online courses require campus classes?'}, {'role': 'assistant', 'content': 'They are delivered entirely online, without required campus attendance.\nSource: https://online.monash.edu/faqs/'}]}, {'id': 'online_study_02', 'group_id': 'online_study_02', 'topic': 'online_study', 'audience': 'Monash Online students', 'question': 'What award do Monash Online graduates receive?', 'answer': 'They receive an accredited Monash qualification equivalent to the corresponding on-campus award.', 'source_url': 'https://online.monash.edu/faqs/', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'validation', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What award do Monash Online graduates receive?'}, {'role': 'assistant', 'content': 'They receive an accredited Monash qualification equivalent to the corresponding on-campus award.\nSource: https://online.monash.edu/faqs/'}]}, {'id': 'online_study_03', 'group_id': 'online_study_03', 'topic': 'online_study', 'audience': 'Monash Online students', 'question': 'Where are Monash Online entry criteria published?', 'answer': 'Each online course page lists its own academic, professional and English requirements.', 'source_url': 'https://online.monash.edu/faqs/', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Where are Monash Online entry criteria published?'}, {'role': 'assistant', 'content': 'Each online course page lists its own academic, professional and English requirements.\nSource: https://online.monash.edu/faqs/'}]}, {'id': 'online_study_04', 'group_id': 'online_study_04', 'topic': 'online_study', 'audience': 'Monash Online students', 'question': 'Where can I check Monash Online tuition costs?', 'answer': 'Check unit fees on the course page; the total depends on your units and approved credit.', 'source_url': 'https://online.monash.edu/faqs/', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Where can I check Monash Online tuition costs?'}, {'role': 'assistant', 'content': 'Check unit fees on the course page; the total depends on your units and approved credit.\nSource: https://online.monash.edu/faqs/'}]}, {'id': 'online_study_05', 'group_id': 'online_study_05', 'topic': 'online_study', 'audience': 'Monash Online students', 'question': 'Are CSP places offered through Monash Online?', 'answer': 'No. The online FAQ says Commonwealth Supported Places are unavailable for these courses.', 'source_url': 'https://online.monash.edu/faqs/', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Are CSP places offered through Monash Online?'}, {'role': 'assistant', 'content': 'No. The online FAQ says Commonwealth Supported Places are unavailable for these courses.\nSource: https://online.monash.edu/faqs/'}]}, {'id': 'online_study_06', 'group_id': 'online_study_06', 'topic': 'online_study', 'audience': 'Monash Online students', 'question': 'How does an online student prepare before classes?', 'answer': 'Set up your account, learn Moodle and complete the Student Hub New Student Checklist.', 'source_url': 'https://online.monash.edu/faqs/', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How does an online student prepare before classes?'}, {'role': 'assistant', 'content': 'Set up your account, learn Moodle and complete the Student Hub New Student Checklist.\nSource: https://online.monash.edu/faqs/'}]}, {'id': 'scholars_program_01', 'group_id': 'scholars_program_01', 'topic': 'scholars_program', 'audience': 'secondary school students', 'question': 'Which school years does Monash Scholars support?', 'answer': 'It supports high-achieving students across Years 10 to 12 with university engagement activities.', 'source_url': 'https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Which school years does Monash Scholars support?'}, {'role': 'assistant', 'content': 'It supports high-achieving students across Years 10 to 12 with university engagement activities.\nSource: https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions'}]}, {'id': 'scholars_program_02', 'group_id': 'scholars_program_02', 'topic': 'scholars_program', 'audience': 'secondary school students', 'question': 'Who can seek entry into Monash Scholars?', 'answer': 'Applications are for endorsed Year 10 students at participating schools.', 'source_url': 'https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Who can seek entry into Monash Scholars?'}, {'role': 'assistant', 'content': 'Applications are for endorsed Year 10 students at participating schools.\nSource: https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions'}]}, {'id': 'scholars_program_03', 'group_id': 'scholars_program_03', 'topic': 'scholars_program', 'audience': 'secondary school students', 'question': 'Does Monash Scholars pay a participation bursary?', 'answer': 'No participation bursary or financial incentive is provided.', 'source_url': 'https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Does Monash Scholars pay a participation bursary?'}, {'role': 'assistant', 'content': 'No participation bursary or financial incentive is provided.\nSource: https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions'}]}, {'id': 'scholars_program_04', 'group_id': 'scholars_program_04', 'topic': 'scholars_program', 'audience': 'secondary school students', 'question': 'Does Monash Scholars participation guarantee university admission?', 'answer': 'No. Participation helps preparation but provides no formal admission advantage.', 'source_url': 'https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Does Monash Scholars participation guarantee university admission?'}, {'role': 'assistant', 'content': 'No. Participation helps preparation but provides no formal admission advantage.\nSource: https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions'}]}, {'id': 'scholars_program_05', 'group_id': 'scholars_program_05', 'topic': 'scholars_program', 'audience': 'secondary school students', 'question': 'Is Monash Scholars participation charged to students?', 'answer': 'There is no participation fee for students.', 'source_url': 'https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Is Monash Scholars participation charged to students?'}, {'role': 'assistant', 'content': 'There is no participation fee for students.\nSource: https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions'}]}, {'id': 'scholars_program_06', 'group_id': 'scholars_program_06', 'topic': 'scholars_program', 'audience': 'secondary school students', 'question': 'Can a Monash Scholar leave the program?', 'answer': 'Yes. Inform the program team when you decide to withdraw.', 'source_url': 'https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can a Monash Scholar leave the program?'}, {'role': 'assistant', 'content': 'Yes. Inform the program team when you decide to withdraw.\nSource: https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions'}]}, {'id': 'research_offers_01', 'group_id': 'research_offers_01', 'topic': 'research_offers', 'audience': 'graduate research applicants', 'question': 'Can research admission be conditional on English proficiency?', 'answer': 'Admission-only conditional offers may be available with faculty support; scholarship offers cannot be conditional on English requirements.', 'source_url': 'https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can research admission be conditional on English proficiency?'}, {'role': 'assistant', 'content': 'Admission-only conditional offers may be available with faculty support; scholarship offers cannot be conditional on English requirements.\nSource: https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers'}]}, {'id': 'research_offers_02', 'group_id': 'research_offers_02', 'topic': 'research_offers', 'audience': 'graduate research applicants', 'question': 'Do research scholarships pay visa application charges?', 'answer': 'No. Those application charges are excluded.', 'source_url': 'https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Do research scholarships pay visa application charges?'}, {'role': 'assistant', 'content': 'No. Those application charges are excluded.\nSource: https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers'}]}, {'id': 'research_offers_03', 'group_id': 'research_offers_03', 'topic': 'research_offers', 'audience': 'graduate research applicants', 'question': 'How are domestic research tuition fees covered?', 'answer': 'The RTP Fees Offset covers course tuition; the Student Services and Amenities fee remains payable.', 'source_url': 'https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How are domestic research tuition fees covered?'}, {'role': 'assistant', 'content': 'The RTP Fees Offset covers course tuition; the Student Services and Amenities fee remains payable.\nSource: https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers'}]}, {'id': 'research_offers_04', 'group_id': 'research_offers_04', 'topic': 'research_offers', 'audience': 'graduate research applicants', 'question': 'Must international research students pay tuition?', 'answer': 'They pay tuition unless supported by a scholarship or external sponsorship.', 'source_url': 'https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Must international research students pay tuition?'}, {'role': 'assistant', 'content': 'They pay tuition unless supported by a scholarship or external sponsorship.\nSource: https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers'}]}, {'id': 'research_offers_05', 'group_id': 'research_offers_05', 'topic': 'research_offers', 'audience': 'graduate research applicants', 'question': 'How can research applicants view scholarship decisions?', 'answer': 'Sign into My.App and view the outcome under My Outcomes/Offers.', 'source_url': 'https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How can research applicants view scholarship decisions?'}, {'role': 'assistant', 'content': 'Sign into My.App and view the outcome under My Outcomes/Offers.\nSource: https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers'}]}, {'id': 'research_offers_06', 'group_id': 'research_offers_06', 'topic': 'research_offers', 'audience': 'graduate research applicants', 'question': 'What follows acceptance of a research scholarship offer?', 'answer': 'Where admission is still pending, the application proceeds through required approvals before a course offer is issued.', 'source_url': 'https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What follows acceptance of a research scholarship offer?'}, {'role': 'assistant', 'content': 'Where admission is still pending, the application proceeds through required approvals before a course offer is issued.\nSource: https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers'}]}, {'id': 'accommodation_01', 'group_id': 'accommodation_01', 'topic': 'accommodation', 'audience': 'on-campus accommodation applicants', 'question': 'Does an on-campus housing application cost anything?', 'answer': 'No application charge applies.', 'source_url': 'https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Does an on-campus housing application cost anything?'}, {'role': 'assistant', 'content': 'No application charge applies.\nSource: https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq'}]}, {'id': 'accommodation_02', 'group_id': 'accommodation_02', 'topic': 'accommodation', 'audience': 'on-campus accommodation applicants', 'question': 'Can I seek campus housing before receiving admission?', 'answer': 'Yes. You can apply before school results or a university course offer.', 'source_url': 'https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'validation', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can I seek campus housing before receiving admission?'}, {'role': 'assistant', 'content': 'Yes. You can apply before school results or a university course offer.\nSource: https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq'}]}, {'id': 'accommodation_03', 'group_id': 'accommodation_03', 'topic': 'accommodation', 'audience': 'on-campus accommodation applicants', 'question': 'Can one housing application select two accommodation sites?', 'answer': 'Select one site; you may request consideration for the same-campus alternative if unsuccessful.', 'source_url': 'https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can one housing application select two accommodation sites?'}, {'role': 'assistant', 'content': 'Select one site; you may request consideration for the same-campus alternative if unsuccessful.\nSource: https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq'}]}, {'id': 'accommodation_04', 'group_id': 'accommodation_04', 'topic': 'accommodation', 'audience': 'on-campus accommodation applicants', 'question': 'Are housing applications always open until a fixed deadline?', 'answer': 'There is no fixed closing date; applications may close when demand is exceptionally high.', 'source_url': 'https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Are housing applications always open until a fixed deadline?'}, {'role': 'assistant', 'content': 'There is no fixed closing date; applications may close when demand is exceptionally high.\nSource: https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq'}]}, {'id': 'accommodation_05', 'group_id': 'accommodation_05', 'topic': 'accommodation', 'audience': 'on-campus accommodation applicants', 'question': 'Can campus housing guarantee allocation near my friends?', 'answer': 'You can request it and each friend should enter matching preferences, but allocation is not guaranteed.', 'source_url': 'https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can campus housing guarantee allocation near my friends?'}, {'role': 'assistant', 'content': 'You can request it and each friend should enter matching preferences, but allocation is not guaranteed.\nSource: https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq'}]}, {'id': 'accommodation_06', 'group_id': 'accommodation_06', 'topic': 'accommodation', 'audience': 'on-campus accommodation applicants', 'question': 'How do I revise submitted accommodation preferences?', 'answer': 'Sign into the Resident Portal and update your preferences.', 'source_url': 'https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How do I revise submitted accommodation preferences?'}, {'role': 'assistant', 'content': 'Sign into the Resident Portal and update your preferences.\nSource: https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq'}]}, {'id': 'library_01', 'group_id': 'library_01', 'topic': 'library', 'audience': 'Victorian campus library users', 'question': 'What identifies me when borrowing at Victorian campus libraries?', 'answer': 'Use a Monash University ID card or a Library membership card.', 'source_url': 'https://www.monash.edu/library/find-and-borrow/borrowing', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What identifies me when borrowing at Victorian campus libraries?'}, {'role': 'assistant', 'content': 'Use a Monash University ID card or a Library membership card.\nSource: https://www.monash.edu/library/find-and-borrow/borrowing'}]}, {'id': 'library_02', 'group_id': 'library_02', 'topic': 'library', 'audience': 'Victorian campus library users', 'question': 'What is the usual undergraduate library borrowing allowance?', 'answer': 'Undergraduates may initially borrow 30 items for two weeks; restricted materials can have different terms.', 'source_url': 'https://www.monash.edu/library/find-and-borrow/borrowing', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'validation', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What is the usual undergraduate library borrowing allowance?'}, {'role': 'assistant', 'content': 'Undergraduates may initially borrow 30 items for two weeks; restricted materials can have different terms.\nSource: https://www.monash.edu/library/find-and-borrow/borrowing'}]}, {'id': 'library_03', 'group_id': 'library_03', 'topic': 'library', 'audience': 'Victorian campus library users', 'question': 'How do I see my current library return dates?', 'answer': 'Sign into Library Search and check My Loans.', 'source_url': 'https://www.monash.edu/library/find-and-borrow/borrowing', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How do I see my current library return dates?'}, {'role': 'assistant', 'content': 'Sign into Library Search and check My Loans.\nSource: https://www.monash.edu/library/find-and-borrow/borrowing'}]}, {'id': 'library_04', 'group_id': 'library_04', 'topic': 'library', 'audience': 'Victorian campus library users', 'question': 'Can books be returned to another Monash campus library?', 'answer': 'Yes. You may return them to any library, regardless of the borrowing location.', 'source_url': 'https://www.monash.edu/library/find-and-borrow/borrowing', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can books be returned to another Monash campus library?'}, {'role': 'assistant', 'content': 'Yes. You may return them to any library, regardless of the borrowing location.\nSource: https://www.monash.edu/library/find-and-borrow/borrowing'}]}, {'id': 'library_05', 'group_id': 'library_05', 'topic': 'library', 'audience': 'Victorian campus library users', 'question': 'Are library book returns available after hours?', 'answer': 'External return chutes allow returns around the clock.', 'source_url': 'https://www.monash.edu/library/find-and-borrow/borrowing', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Are library book returns available after hours?'}, {'role': 'assistant', 'content': 'External return chutes allow returns around the clock.\nSource: https://www.monash.edu/library/find-and-borrow/borrowing'}]}, {'id': 'library_06', 'group_id': 'library_06', 'topic': 'library', 'audience': 'Victorian campus library users', 'question': 'What changes when a library loan becomes overdue?', 'answer': 'Further borrowing is blocked and other loans stop renewing automatically.', 'source_url': 'https://www.monash.edu/library/find-and-borrow/borrowing', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What changes when a library loan becomes overdue?'}, {'role': 'assistant', 'content': 'Further borrowing is blocked and other loans stop renewing automatically.\nSource: https://www.monash.edu/library/find-and-borrow/borrowing'}]}, {'id': 'graduate_study_01', 'group_id': 'graduate_study_01', 'topic': 'graduate_study', 'audience': 'graduate coursework applicants', 'question': 'Can graduate study be taken in flexible modes?', 'answer': 'Depending on the course, part-time, online or intensive options may be available; check its course entry.', 'source_url': 'https://www.monash.edu/graduate-study/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can graduate study be taken in flexible modes?'}, {'role': 'assistant', 'content': 'Depending on the course, part-time, online or intensive options may be available; check its course entry.\nSource: https://www.monash.edu/graduate-study/faqs'}]}, {'id': 'graduate_study_02', 'group_id': 'graduate_study_02', 'topic': 'graduate_study', 'audience': 'graduate coursework applicants', 'question': 'Can I apply while finishing my undergraduate degree?', 'answer': 'Yes. Submit your latest transcript; an offer may depend on completing your degree successfully.', 'source_url': 'https://www.monash.edu/graduate-study/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can I apply while finishing my undergraduate degree?'}, {'role': 'assistant', 'content': 'Yes. Submit your latest transcript; an offer may depend on completing your degree successfully.\nSource: https://www.monash.edu/graduate-study/faqs'}]}, {'id': 'graduate_study_03', 'group_id': 'graduate_study_03', 'topic': 'graduate_study', 'audience': 'graduate coursework applicants', 'question': 'Does a masters always need the same undergraduate discipline?', 'answer': 'No. Some courses accept other disciplines, while others require particular backgrounds or prerequisites.', 'source_url': 'https://www.monash.edu/graduate-study/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Does a masters always need the same undergraduate discipline?'}, {'role': 'assistant', 'content': 'No. Some courses accept other disciplines, while others require particular backgrounds or prerequisites.\nSource: https://www.monash.edu/graduate-study/faqs'}]}, {'id': 'graduate_study_04', 'group_id': 'graduate_study_04', 'topic': 'graduate_study', 'audience': 'graduate coursework applicants', 'question': 'What are course maps used for?', 'answer': 'They visually outline course structure and help you choose units and electives.', 'source_url': 'https://www.monash.edu/graduate-study/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'validation', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What are course maps used for?'}, {'role': 'assistant', 'content': 'They visually outline course structure and help you choose units and electives.\nSource: https://www.monash.edu/graduate-study/faqs'}]}, {'id': 'graduate_study_05', 'group_id': 'graduate_study_05', 'topic': 'graduate_study', 'audience': 'graduate coursework applicants', 'question': 'When can I inspect my graduate unit timetable?', 'answer': 'Relevant timetables become available through Allocate+ after enrolment.', 'source_url': 'https://www.monash.edu/graduate-study/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'When can I inspect my graduate unit timetable?'}, {'role': 'assistant', 'content': 'Relevant timetables become available through Allocate+ after enrolment.\nSource: https://www.monash.edu/graduate-study/faqs'}]}, {'id': 'graduate_study_06', 'group_id': 'graduate_study_06', 'topic': 'graduate_study', 'audience': 'graduate coursework applicants', 'question': 'How much weekly study is expected for full-time coursework?', 'answer': 'A standard load of four six-credit-point units is described as roughly 48 hours per week.', 'source_url': 'https://www.monash.edu/graduate-study/faqs', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How much weekly study is expected for full-time coursework?'}, {'role': 'assistant', 'content': 'A standard load of four six-credit-point units is described as roughly 48 hours per week.\nSource: https://www.monash.edu/graduate-study/faqs'}]}, {'id': 'scholarship_applications_01', 'group_id': 'scholarship_applications_01', 'topic': 'scholarship_applications', 'audience': 'scholarship applicants; rules vary by applicant type', 'question': 'Can international applicants seek Monash equity scholarships?', 'answer': 'No. The scholarship application page excludes international applicants from equity scholarships.', 'source_url': 'https://www.monash.edu/study/fees-scholarships/scholarships/applications', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can international applicants seek Monash equity scholarships?'}, {'role': 'assistant', 'content': 'No. The scholarship application page excludes international applicants from equity scholarships.\nSource: https://www.monash.edu/study/fees-scholarships/scholarships/applications'}]}, {'id': 'scholarship_applications_02', 'group_id': 'scholarship_applications_02', 'topic': 'scholarship_applications', 'audience': 'scholarship applicants; rules vary by applicant type', 'question': 'Can every Monash scholarship be deferred?', 'answer': 'No. Check the individual award; deferral is available only where its conditions permit.', 'source_url': 'https://www.monash.edu/study/fees-scholarships/scholarships/applications', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can every Monash scholarship be deferred?'}, {'role': 'assistant', 'content': 'No. Check the individual award; deferral is available only where its conditions permit.\nSource: https://www.monash.edu/study/fees-scholarships/scholarships/applications'}]}, {'id': 'scholarship_applications_03', 'group_id': 'scholarship_applications_03', 'topic': 'scholarship_applications', 'audience': 'scholarship applicants; rules vary by applicant type', 'question': 'Which office handles research scholarship applications?', 'answer': 'Applications for research masters and doctorate scholarships go through Monash Graduate Education.', 'source_url': 'https://www.monash.edu/study/fees-scholarships/scholarships/applications', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Which office handles research scholarship applications?'}, {'role': 'assistant', 'content': 'Applications for research masters and doctorate scholarships go through Monash Graduate Education.\nSource: https://www.monash.edu/study/fees-scholarships/scholarships/applications'}]}, {'id': 'scholarship_applications_04', 'group_id': 'scholarship_applications_04', 'topic': 'scholarship_applications', 'audience': 'scholarship applicants; rules vary by applicant type', 'question': 'What file format is used for scholarship supporting documents?', 'answer': 'Upload PDF documents at the specified application step; posted paper copies are not accepted.', 'source_url': 'https://www.monash.edu/study/fees-scholarships/scholarships/applications', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What file format is used for scholarship supporting documents?'}, {'role': 'assistant', 'content': 'Upload PDF documents at the specified application step; posted paper copies are not accepted.\nSource: https://www.monash.edu/study/fees-scholarships/scholarships/applications'}]}, {'id': 'scholarship_applications_05', 'group_id': 'scholarship_applications_05', 'topic': 'scholarship_applications', 'audience': 'scholarship applicants; rules vary by applicant type', 'question': 'Does the KC Kuok award need a separate application?', 'answer': 'Yes. Submit its separate application form.', 'source_url': 'https://www.monash.edu/study/fees-scholarships/scholarships/applications', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Does the KC Kuok award need a separate application?'}, {'role': 'assistant', 'content': 'Yes. Submit its separate application form.\nSource: https://www.monash.edu/study/fees-scholarships/scholarships/applications'}]}, {'id': 'scholarship_applications_06', 'group_id': 'scholarship_applications_06', 'topic': 'scholarship_applications', 'audience': 'scholarship applicants; rules vary by applicant type', 'question': 'How can a submitted scholarship application be changed?', 'answer': 'Request deletion of the original application online, then submit a new application.', 'source_url': 'https://www.monash.edu/study/fees-scholarships/scholarships/applications', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How can a submitted scholarship application be changed?'}, {'role': 'assistant', 'content': 'Request deletion of the original application online, then submit a new application.\nSource: https://www.monash.edu/study/fees-scholarships/scholarships/applications'}]}, {'id': 'international_postgraduate_01', 'group_id': 'international_postgraduate_01', 'topic': 'international_postgraduate', 'audience': 'international postgraduate applicants', 'question': 'How are international postgraduate tuition amounts determined?', 'answer': 'Fees depend on your course and commencement year; consult course pages or the fee calculator.', 'source_url': 'https://www.monash.edu/study/international/graduate', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'test', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How are international postgraduate tuition amounts determined?'}, {'role': 'assistant', 'content': 'Fees depend on your course and commencement year; consult course pages or the fee calculator.\nSource: https://www.monash.edu/study/international/graduate'}]}, {'id': 'international_postgraduate_02', 'group_id': 'international_postgraduate_02', 'topic': 'international_postgraduate', 'audience': 'international postgraduate applicants', 'question': 'Can overseas funding help pay for Monash postgraduate study?', 'answer': 'Your home country or a United Nations program may offer support; consult foreign financial aid information.', 'source_url': 'https://www.monash.edu/study/international/graduate', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'Can overseas funding help pay for Monash postgraduate study?'}, {'role': 'assistant', 'content': 'Your home country or a United Nations program may offer support; consult foreign financial aid information.\nSource: https://www.monash.edu/study/international/graduate'}]}, {'id': 'international_postgraduate_03', 'group_id': 'international_postgraduate_03', 'topic': 'international_postgraduate', 'audience': 'international postgraduate applicants', 'question': 'How can international coursework applicants submit an application?', 'answer': 'Apply directly through the Monash application portal or through an authorised education agent.', 'source_url': 'https://www.monash.edu/study/international/graduate', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'How can international coursework applicants submit an application?'}, {'role': 'assistant', 'content': 'Apply directly through the Monash application portal or through an authorised education agent.\nSource: https://www.monash.edu/study/international/graduate'}]}, {'id': 'international_postgraduate_04', 'group_id': 'international_postgraduate_04', 'topic': 'international_postgraduate', 'audience': 'international postgraduate applicants', 'question': 'What documents accompany international postgraduate coursework applications?', 'answer': 'Provide academic transcripts, English proficiency evidence and a passport copy.', 'source_url': 'https://www.monash.edu/study/international/graduate', 'observed_date': '2026-10-08', 'language': 'en', 'transformation': 'source_grounded_summary', 'split': 'train', 'messages': [{'role': 'system', 'content': 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'}, {'role': 'user', 'content': 'What documents accompany international postgraduate coursework applications?'}, {'role': 'assistant', 'content': 'Provide academic transcripts, English proficiency evidence and a passport copy.\nSource: https://www.monash.edu/study/international/graduate'}]}]
sources = [{'source_id': 'international_application', 'title': 'International application FAQ', 'url': 'https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq', 'observed_date': '2026-10-08', 'audience': 'international applicants', 'record_count': 6, 'summary_word_count': 121, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'vtac_preferences', 'title': 'Change of preference FAQ', 'url': 'https://www.monash.edu/change-of-preference/faqs', 'observed_date': '2026-10-08', 'audience': 'VTAC applicants', 'record_count': 6, 'summary_word_count': 131, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'online_study', 'title': 'Monash Online FAQ', 'url': 'https://online.monash.edu/faqs/', 'observed_date': '2026-10-08', 'audience': 'Monash Online students', 'record_count': 6, 'summary_word_count': 120, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'scholars_program', 'title': 'Monash Scholars FAQ', 'url': 'https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions', 'observed_date': '2026-10-08', 'audience': 'secondary school students', 'record_count': 6, 'summary_word_count': 100, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'research_offers', 'title': 'Graduate research admission and scholarship FAQ', 'url': 'https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers', 'observed_date': '2026-10-08', 'audience': 'graduate research applicants', 'record_count': 6, 'summary_word_count': 119, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'accommodation', 'title': 'On-campus accommodation application FAQ', 'url': 'https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq', 'observed_date': '2026-10-08', 'audience': 'on-campus accommodation applicants', 'record_count': 6, 'summary_word_count': 115, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'library', 'title': 'Library borrowing', 'url': 'https://www.monash.edu/library/find-and-borrow/borrowing', 'observed_date': '2026-10-08', 'audience': 'Victorian campus library users', 'record_count': 6, 'summary_word_count': 115, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'graduate_study', 'title': 'Graduate study FAQ', 'url': 'https://www.monash.edu/graduate-study/faqs', 'observed_date': '2026-10-08', 'audience': 'graduate coursework applicants', 'record_count': 6, 'summary_word_count': 126, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'scholarship_applications', 'title': 'Scholarship applications', 'url': 'https://www.monash.edu/study/fees-scholarships/scholarships/applications', 'observed_date': '2026-10-08', 'audience': 'scholarship applicants; rules vary by applicant type', 'record_count': 6, 'summary_word_count': 113, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}, {'source_id': 'international_postgraduate', 'title': 'International postgraduate study', 'url': 'https://www.monash.edu/study/international/graduate', 'observed_date': '2026-10-08', 'audience': 'international postgraduate applicants', 'record_count': 4, 'summary_word_count': 85, 'acquisition': 'Public page content inspected via web search/open tools', 'transformation': 'Short, independently phrased English question-answer summaries; not a full-page export'}]
SYSTEM_PROMPT = 'You are a Monash University FAQ study assistant. Give a brief English answer, respect the applicant type and study mode, and cite the official source URL when known. Do not invent course-specific requirements, prices, deadlines or personal application outcomes. If the information is unknown, explain that and direct the user to the relevant official Monash team. Your training snapshot is dated 2026-10-08; policies may change. You are not an official university representative.'


In [ ]:
assert len(records) == len({r['id'] for r in records})
assert len(records) == len({r['question'].strip().lower() for r in records})
assert all(r['answer'] and r['source_url'].startswith('https://') for r in records)
group_splits = {}
for r in records:
    assert [m['role'] for m in r['messages']] == ['system', 'user', 'assistant']
    assert r['messages'][1]['content'] == r['question']
    if r['group_id'] in group_splits:
        assert group_splits[r['group_id']] == r['split'], 'The same fact appears in multiple splits'
    group_splits[r['group_id']] = r['split']
splits = {s: [r for r in records if r['split'] == s] for s in ('train', 'validation', 'test')}
for name, items in splits.items():
    (DATA_DIR / f'{name}.jsonl').write_text(
        ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in items), encoding='utf-8')
print('Split counts:', {s: len(items) for s, items in splits.items()})
print('Topics:', dict(Counter(r['topic'] for r in records)))
for source in sources:
    print(source['source_id'], source['url'])
display({k: records[0][k] for k in ('question', 'answer', 'source_url', 'audience', 'split')})


## 3. Optional: refresh public-page candidates

The reviewed dataset is already embedded, so this step is disabled by default. Set REFRESH_PUBLIC_PAGES to True and place collect_monash_candidates.py beside the Notebook to refresh candidates. The collector checks robots.txt, limits requests to official domains and spaces requests by two seconds. It skips access denials and never bypasses them.

During preparation, ordinary HTTP access returned 403; the embedded summaries were checked against accessible official web-search/page content. Automatically extracted candidates need review for answer boundaries, scope and date conditions before they enter training.


In [ ]:
REFRESH_PUBLIC_PAGES = False
if REFRESH_PUBLIC_PAGES:
    import runpy
    collector = Path('collect_monash_candidates.py')
    if not collector.exists():
        raise FileNotFoundError('Place collect_monash_candidates.py beside this Notebook to refresh public-page candidates.')
    collector_namespace = runpy.run_path(str(collector))
    result = collector_namespace['collect'](sources, ROOT / 'raw_candidates')
    print(result)
else:
    print('Using the reviewed embedded summary dataset.')


## 4. Check the runtime and configure training

The default is Qwen/Qwen2.5-1.5B-Instruct with 4-bit NF4 and LoRA rank 16. Parameters are pilot starting values; measure memory and runtime on the selected GPU. No API keys are needed. The Notebook does not publish the model or dataset.


In [ ]:
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    Trainer, TrainingArguments, EarlyStoppingCallback, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from torch.utils.data import Dataset
set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('No NVIDIA CUDA GPU found. Enable a Kaggle GPU accelerator; data checks can run independently.')
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_LENGTH = 512
EPOCHS = 3
LEARNING_RATE = 1e-4
BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
USE_CUDA_FP16_AMP = False  # k-bit preparation casts some parameters to float32; avoid fp16 gradient-scaling conflicts


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config,
    device_map={'': 0}, torch_dtype=COMPUTE_DTYPE, attn_implementation='eager')
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False})
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules='all-linear', bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 5. Compute loss only on assistant answers

Use the model's chat template. Mask system/user labels and padding with -100. Reject oversized records instead of silently truncating the question or all answer tokens.


In [ ]:
def encode_record(record, tokenizer, max_length):
    messages = record['messages']
    prompt_ids = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)
    full_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
    assert full_ids[:len(prompt_ids)] == prompt_ids, 'Chat-template prefixes differ; inspect the assistant-answer boundary'
    assert len(full_ids) <= max_length, f"Record {record['id']} exceeds the sequence limit: {len(full_ids)}"
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    assert len(full_ids) > len(prompt_ids), 'The assistant answer is empty'
    assert all(x == -100 for x in labels[:len(prompt_ids)])
    return {'input_ids': full_ids, 'attention_mask': [1] * len(full_ids), 'labels': labels}

class FAQDataset(Dataset):
    def __init__(self, items):
        self.items = [encode_record(r, tokenizer, MAX_LENGTH) for r in items]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        return self.items[index]

def collate_assistant_only(features):
    length = max(len(f['input_ids']) for f in features)
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for f in features:
        padding = length - len(f['input_ids'])
        batch['input_ids'].append(f['input_ids'] + [tokenizer.pad_token_id] * padding)
        batch['attention_mask'].append(f['attention_mask'] + [0] * padding)
        batch['labels'].append(f['labels'] + [-100] * padding)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

train_ds = FAQDataset(splits['train'])
val_ds = FAQDataset(splits['validation'])
test_ds = FAQDataset(splits['test'])
print('Maximum sequence length:', max(len(encode_record(r, tokenizer, MAX_LENGTH)['input_ids']) for r in records))
probe = collate_assistant_only([train_ds[0], train_ds[1]])
assert torch.all(probe['labels'][probe['attention_mask'] == 0] == -100)
example = train_ds[0]
display(tokenizer.decode([t for t in example['labels'] if t != -100]))


## 6. Record the baseline and train

Compare identical held-out test questions before and after training. Inference receives only the question and system prompt. Validation runs after each epoch and selects the checkpoint with the lowest validation loss.


In [ ]:
def ask(question, max_new_tokens=160):
    was_training = model.training
    model.eval()
    prompt = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': question}]
    ids = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True,
        return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids),
            max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            use_cache=True)
    response = tokenizer.decode(output[0, ids.shape[1]:], skip_special_tokens=True)
    if was_training:
        model.train()
    return response

args = TrainingArguments(output_dir=str(ROOT / 'checkpoints'), num_train_epochs=EPOCHS,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=LEARNING_RATE, lr_scheduler_type='cosine',
    warmup_ratio=0.1, optim='adamw_torch', weight_decay=0.01, max_grad_norm=1.0,
    logging_steps=2, eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    bf16=BF16, fp16=USE_CUDA_FP16_AMP, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none', seed=SEED, data_seed=SEED, dataloader_num_workers=0,
    remove_unused_columns=False, label_names=['labels'], prediction_loss_only=True)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collate_assistant_only, processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
with model.disable_adapter():
    baseline_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='baseline_test')
    baseline_answers = {r['id']: ask(r['question']) for r in splits['test']}
(ROOT / 'baseline.json').write_text(json.dumps(
    {'metrics': baseline_metrics, 'answers': baseline_answers}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Baseline test loss:', baseline_metrics['baseline_test_loss'])


In [ ]:
train_result = trainer.train()
print(train_result.metrics)
print('Best checkpoint:', trainer.state.best_model_checkpoint)


## 7. Evaluate and create the manual review table

A lower loss does not prove factual accuracy. The CSV includes manual scoring columns for factual correctness, audience scope, URLs and time conditions. Ten test questions provide only a coarse check; the Notebook does not declare an accuracy target reached.


In [ ]:
after_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='finetuned_test')
comparison = []
for r in splits['test']:
    comparison.append({'id': r['id'], 'topic': r['topic'], 'question': r['question'],
        'reference_answer': r['answer'], 'source_url': r['source_url'],
        'before': baseline_answers[r['id']], 'after': ask(r['question']),
        'before_correct_0_or_1': '', 'after_correct_0_or_1': '', 'review_notes': ''})
comparison_path = ROOT / 'before_after_review.csv'
with comparison_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)
metrics = {'baseline_test': baseline_metrics, 'finetuned_test': after_metrics,
    'train': train_result.metrics, 'test_size': len(test_ds),
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'note': 'Loss is not factual accuracy; use the manual review CSV.'}
(ROOT / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
display(metrics)
display(comparison[0])


## 8. Save the adapter and run manifest

The exported LoRA adapter requires the original base model. The manifest records the dataset hash, model revision, dependency versions and training parameters. No files are uploaded to Hugging Face.


In [ ]:
import importlib.metadata
ADAPTER_DIR = ROOT / 'monash_qa_adapter'
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()
versions = {name: importlib.metadata.version(name) for name in
    ('torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes')}
data_hash = hashlib.sha256(json.dumps(records, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
manifest = {'model_id': MODEL_ID, 'model_revision': getattr(model.config, '_commit_hash', None),
    'source_snapshot_date': '2026-10-08', 'data_sha256': data_hash,
    'versions': versions, 'seed': SEED, 'max_length': MAX_LENGTH,
    'counts': {s: len(v) for s, v in splits.items()}, 'training_args': args.to_dict(),
    'lora': {'r': 16, 'alpha': 32, 'dropout': 0.05, 'target_modules': 'all-linear'},
    'sources': sources}
(ROOT / 'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2,
    default=str), encoding='utf-8')
print('Adapter directory:', ADAPTER_DIR)


## 9. Reload the exported adapter

Release the training model and reload the saved adapter to check that the exported files can restore inference. Use ask(...) after this section.


In [ ]:
import gc
del trainer, model
gc.collect()
torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config,
    device_map={'': 0}, torch_dtype=COMPUTE_DTYPE, attn_implementation='eager')
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR), is_trainable=False)
model.eval()
print(ask('Does an on-campus housing application cost anything?'))


In [ ]:
print(ask('What identifies me when borrowing at Victorian campus libraries?'))


## 10. Export training artifacts

Package the adapter, test comparisons, dataset and manifest without intermediate checkpoints. The Kaggle archive is /kaggle/working/monash_trained_adapter.zip and is available through notebook outputs. Colab provides a download prompt.


In [ ]:
import zipfile
archive_path = ROOT.parent / 'monash_trained_adapter.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in (ADAPTER_DIR, DATA_DIR):
        for path in folder.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to(ROOT))
    for name in ('metrics.json', 'baseline.json', 'before_after_review.csv', 'run_manifest.json'):
        archive.write(ROOT / name, name)
print('Training archive:', archive_path)
from IPython.display import FileLink
display(FileLink(str(archive_path.relative_to(Path.cwd()))))


## Extending the pilot

Review official evidence before adding records. Assign the same group_id to paraphrases or translations of the same fact, then split by group. Exclude navigation, marketing copy and answers missing necessary context.

This pilot has no real-time website retrieval and does not establish refusal quality on unknown questions. Refresh evidence when policies, fees or dates change. Larger reviewed training and test sets are needed to assess broader coverage.


In [ ]:
import sys, types
retrieval_module = types.ModuleType("retrieval")
exec('"""Auditable BM25 retrieval over a frozen, paraphrased FAQ snapshot."""\nimport json\nimport math\nimport re\nfrom collections import Counter\nfrom pathlib import Path\n\nSTOP = set(\'a an the i my me we you your it is are was be do does did to of for in on at and or can how what where when will have may with as each\'.split())\nREFUSAL = \'I cannot establish that from the supplied Monash FAQ snapshot. Please check with the relevant official Monash team.\'\n\ndef tokens(text):\n    return [word for word in re.findall(r\'[a-z0-9]+\', text.lower()) if word not in STOP]\n\ndef read_jsonl(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n\nclass BM25:\n    def __init__(self, records, k1=1.5, b=0.75):\n        if not records:\n            raise ValueError(\'The knowledge base is empty\')\n        self.records = records\n        self.k1, self.b = k1, b\n        self.docs = [Counter(tokens(r[\'question\'] + \' \' + r[\'answer\'])) for r in records]\n        self.lengths = [sum(d.values()) for d in self.docs]\n        self.avg_length = sum(self.lengths) / len(records)\n        df = Counter(t for doc in self.docs for t in doc)\n        self.idf = {t: math.log(1 + (len(records) - n + .5)/(n + .5)) for t, n in df.items()}\n\n    def search(self, query, top_k=3):\n        terms = set(tokens(query))\n        hits = []\n        for record, doc, length in zip(self.records, self.docs, self.lengths):\n            score = sum(self.idf.get(t, 0) * doc.get(t, 0) * (self.k1+1) /\n                        (doc.get(t, 0) + self.k1*(1-self.b+self.b*length/self.avg_length)) for t in terms)\n            # Coverage is bounded and includes query terms absent from the corpus.\n            coverage = len(terms.intersection(doc)) / max(1, len(terms))\n            hits.append({\'id\': record[\'id\'], \'score\': score, \'coverage\': coverage, \'record\': record})\n        return sorted(hits, key=lambda h: (-h[\'score\'], h[\'id\']))[:top_k]\n\ndef calibrate(index, cases):\n    candidates = [x/20 for x in range(21)] + [1.01]\n    details = []\n    for threshold in candidates:\n        correct = 0\n        for case in cases:\n            hit = index.search(case[\'question\'], 1)[0]\n            accepted = hit[\'score\'] > 0 and hit[\'coverage\'] >= threshold\n            correct += ((accepted and hit[\'id\'] == case[\'target_id\']) if case[\'kind\'] == \'answerable\' else not accepted)\n        details.append({\'threshold\': threshold, \'correct_decisions\': int(correct), \'total\': len(cases)})\n    best = max(details, key=lambda row: (row[\'correct_decisions\'], row[\'threshold\']))\n    return {\'threshold\': best[\'threshold\'], \'selection\': \'Maximize calibration decision count; ties prefer greater abstention.\',\n            \'calibration_only\': True, \'candidates\': details}\n\ndef retrieve(index, question, threshold):\n    hits = index.search(question, 3)\n    accepted = hits[0][\'score\'] > 0 and hits[0][\'coverage\'] >= threshold\n    return hits, accepted\n\ndef context_messages(question, hits):\n    context = \'\\n\\n\'.join(f"[{n}] Audience: {h[\'record\'][\'audience\']}\\nQuestion: {h[\'record\'][\'question\']}\\nAnswer: {h[\'record\'][\'answer\']}" for n,h in enumerate(hits, 1))\n    return [\n        {\'role\': \'system\', \'content\': \'Answer briefly in English using only the supplied FAQ excerpts. Respect their audience. If they do not establish the requested fact, say you cannot establish it from this snapshot. Do not invent numbers, dates, personal outcomes, or URLs. Do not output source URLs; the application attaches retrieval metadata separately. Snapshot date: 2026-10-08. You are not an official university representative.\'},\n        {\'role\': \'user\', \'content\': f\'FAQ excerpts:\\n{context}\\n\\nUser question: {question}\'},\n    ]\n\ndef is_refusal(text):\n    # A surface heuristic, never used as a factual correctness label.\n    phrases = [\'cannot establish\', \'cannot determine\', "can\'t determine", \'not provided\',\n               \'not available\', \'not specified\', \'not included\', \'does not specify\',\n               \'do not have access\', "don\'t have access", \'cannot access\', \'not contain\',\n               \'insufficient information\', \'no information\', \'cannot confirm\', \'cannot guarantee\']\n    return any(phrase in text.lower() for phrase in phrases)\n\ndef source_urls(text):\n    return [s.rstrip(\'.,);]\') for s in re.findall(r\'https?://[^\\s<>]+\', text)]\n', retrieval_module.__dict__)
sys.modules["retrieval"] = retrieval_module
benchmark_cases = [{'id': 'heldout_international_application_04', 'kind': 'answerable', 'question': 'I applied as an international student and now want another course. How do I request a course change?', 'target_id': 'international_application_04', 'reference_answer': 'Send the request to mu.documents@monash.edu.', 'source_url': 'https://www.monash.edu/study/how-to-apply/application-process/international-students/after-you-apply/faq', 'topic': 'international_application'}, {'id': 'heldout_vtac_preferences_04', 'kind': 'answerable', 'question': 'Do I have to pay each time I reorder my VTAC course preferences?', 'target_id': 'vtac_preferences_04', 'reference_answer': 'Changes are free and can be repeated before the relevant deadline.', 'source_url': 'https://www.monash.edu/change-of-preference/faqs', 'topic': 'vtac_preferences'}, {'id': 'heldout_online_study_04', 'kind': 'answerable', 'question': 'How can I work out the tuition price of a Monash Online course?', 'target_id': 'online_study_04', 'reference_answer': 'Check unit fees on the course page; the total depends on your units and approved credit.', 'source_url': 'https://online.monash.edu/faqs/', 'topic': 'online_study'}, {'id': 'heldout_scholars_program_02', 'kind': 'answerable', 'question': 'What school year and school endorsement are needed to apply for Monash Scholars?', 'target_id': 'scholars_program_02', 'reference_answer': 'Applications are for endorsed Year 10 students at participating schools.', 'source_url': 'https://www.monash.edu/study/why-choose-monash/information-for-high-achieving-students/monash-scholars/monash-scholars-frequently-asked-questions', 'topic': 'scholars_program'}, {'id': 'heldout_research_offers_02', 'kind': 'answerable', 'question': 'Will a graduate research scholarship cover my visa application fee?', 'target_id': 'research_offers_02', 'reference_answer': 'No. Those application charges are excluded.', 'source_url': 'https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship/application,-admission-and-scholarship-offers', 'topic': 'research_offers'}, {'id': 'heldout_accommodation_03', 'kind': 'answerable', 'question': 'May I list two accommodation sites in the same campus housing application?', 'target_id': 'accommodation_03', 'reference_answer': 'Select one site; you may request consideration for the same-campus alternative if unsuccessful.', 'source_url': 'https://www.monash.edu/accommodation/on-campus-accommodation/apply/faq', 'topic': 'accommodation'}, {'id': 'heldout_library_06', 'kind': 'answerable', 'question': 'If a Victorian campus library item is overdue, can I still borrow and have my other loans auto-renew?', 'target_id': 'library_06', 'reference_answer': 'Further borrowing is blocked and other loans stop renewing automatically.', 'source_url': 'https://www.monash.edu/library/find-and-borrow/borrowing', 'topic': 'library'}, {'id': 'heldout_graduate_study_05', 'kind': 'answerable', 'question': 'After enrolling in graduate coursework, where do I see my unit timetable?', 'target_id': 'graduate_study_05', 'reference_answer': 'Relevant timetables become available through Allocate+ after enrolment.', 'source_url': 'https://www.monash.edu/graduate-study/faqs', 'topic': 'graduate_study'}, {'id': 'heldout_scholarship_applications_03', 'kind': 'answerable', 'question': 'Where should I submit a research masters or PhD scholarship application?', 'target_id': 'scholarship_applications_03', 'reference_answer': 'Applications for research masters and doctorate scholarships go through Monash Graduate Education.', 'source_url': 'https://www.monash.edu/study/fees-scholarships/scholarships/applications', 'topic': 'scholarship_applications'}, {'id': 'heldout_international_postgraduate_01', 'kind': 'answerable', 'question': 'What determines international postgraduate course fees, and where should I check them?', 'target_id': 'international_postgraduate_01', 'reference_answer': 'Fees depend on your course and commencement year; consult course pages or the fee calculator.', 'source_url': 'https://www.monash.edu/study/international/graduate', 'topic': 'international_postgraduate'}, {'id': 'unsupported_01', 'kind': 'unsupported', 'question': 'What is my personal Monash application status today?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'international_application'}, {'id': 'unsupported_02', 'kind': 'unsupported', 'question': 'What is the exact closing date for VTAC changes in 2028?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'vtac_preferences'}, {'id': 'unsupported_03', 'kind': 'unsupported', 'question': 'What is the exact total 2028 tuition price for my Monash Online MBA?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'online_study'}, {'id': 'unsupported_04', 'kind': 'unsupported', 'question': 'Am I personally guaranteed a place in Monash Scholars?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'scholars_program'}, {'id': 'unsupported_05', 'kind': 'unsupported', 'question': 'What will my Australian visa application cost in 2028?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'research_offers'}, {'id': 'unsupported_06', 'kind': 'unsupported', 'question': 'Which exact room number will I be assigned next semester?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'accommodation'}, {'id': 'unsupported_07', 'kind': 'unsupported', 'question': 'How many books are currently checked out on my personal library account?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'library'}, {'id': 'unsupported_08', 'kind': 'unsupported', 'question': 'What is my individual graduate class timetable for next Tuesday?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'graduate_study'}, {'id': 'unsupported_09', 'kind': 'unsupported', 'question': 'Will my personal PhD scholarship application be successful?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'scholarship_applications'}, {'id': 'unsupported_10', 'kind': 'unsupported', 'question': 'What exact postgraduate tuition will I owe after my individual credit assessment?', 'target_id': None, 'reference_answer': 'The supplied snapshot cannot establish this information.', 'source_url': None, 'topic': 'international_postgraduate'}]
calibration_cases = [{'id': 'cal_international_application_02', 'kind': 'answerable', 'question': 'What evidence might I need to satisfy a conditional admission offer?', 'target_id': 'international_application_02'}, {'id': 'cal_online_study_02', 'kind': 'answerable', 'question': 'Is the qualification awarded through Monash Online equivalent to the on-campus award?', 'target_id': 'online_study_02'}, {'id': 'cal_accommodation_02', 'kind': 'answerable', 'question': 'Do I need a course offer before applying for campus accommodation?', 'target_id': 'accommodation_02'}, {'id': 'cal_library_02', 'kind': 'answerable', 'question': 'How many ordinary library items may an undergraduate initially borrow and for how long?', 'target_id': 'library_02'}, {'id': 'cal_graduate_study_04', 'kind': 'answerable', 'question': 'How does a graduate course map help me plan my units?', 'target_id': 'graduate_study_04'}, {'id': 'cal_unsupported_01', 'kind': 'unsupported', 'question': 'What exact IELTS score is shown on my personal conditional offer?', 'target_id': None}, {'id': 'cal_unsupported_02', 'kind': 'unsupported', 'question': 'What grade will I get in my next Monash Online exam?', 'target_id': None}, {'id': 'cal_unsupported_03', 'kind': 'unsupported', 'question': 'What is my accommodation waitlist position today?', 'target_id': None}, {'id': 'cal_unsupported_04', 'kind': 'unsupported', 'question': 'What is the due date of the book I borrowed yesterday?', 'target_id': None}, {'id': 'cal_unsupported_05', 'kind': 'unsupported', 'question': 'Will my individual graduate credit transfer request be approved?', 'target_id': None}]
"""Run after the training notebook has reloaded its exported PEFT adapter.

Required globals: model, tokenizer, records, SYSTEM_PROMPT, ROOT, SEED,
benchmark_cases, calibration_cases. The standalone notebook supplies them.
"""
import contextlib
import hashlib
import importlib.metadata
import json
import platform
import time
import zipfile
from pathlib import Path
import torch
from retrieval import BM25, calibrate, context_messages, is_refusal, source_urls, REFUSAL

COMPARISON_DIR = ROOT.parent / 'monash_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
index = BM25(records)
calibration_result = calibrate(index, calibration_cases)
threshold = calibration_result['threshold']

def generate_answer(messages, adapter_enabled):
    ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                         return_tensors='pt').to(model.device)
    if ids.shape[1] > 1024:
        raise ValueError('Prompt exceeds preregistered limit; no silent truncation')
    manager = contextlib.nullcontext() if adapter_enabled else model.disable_adapter()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with manager, torch.inference_mode():
        output = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids),
                                do_sample=False, max_new_tokens=160, use_cache=True,
                                pad_token_id=tokenizer.pad_token_id,
                                eos_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    answer = tokenizer.decode(output[0, ids.shape[1]:], skip_special_tokens=True)
    return answer, time.perf_counter()-started, int(output.shape[1]-ids.shape[1])

model.eval()
torch.cuda.reset_peak_memory_stats()
predictions = []
for case in benchmark_cases:
    hits = index.search(case['question'], 3)
    accepted = hits[0]['score'] > 0 and hits[0]['coverage'] >= threshold
    closed = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':case['question']}]
    for method in ['base', 'qlora', 'rag_base', 'rag_qlora']:
        rag = method.startswith('rag_')
        answer, seconds, count = generate_answer(context_messages(case['question'], hits) if rag else closed,
                                                  method in ['qlora','rag_qlora'])
        urls = sorted({h['record']['source_url'] for h in hits}) if rag else []
        row = {**case, 'method':method, 'answer_body':answer, 'retrieved_ids':[h['id'] for h in hits] if rag else [],
               'candidate_source_urls': urls, 'top1_coverage':hits[0]['coverage'] if rag else None,
               'gate_accepted':accepted if rag else None, 'refusal_heuristic':is_refusal(answer),
               'generated_urls':source_urls(answer), 'seconds':seconds, 'output_tokens':count,
               'manual_correct':None, 'manual_refusal':None, 'manual_notes':None}
        predictions.append(row)
        # This is a post-processing ablation of exactly the same RAG output.
        if method=='rag_base':
            predictions.append({**row, 'method':'gated_rag_base',
                                'answer_body':answer if accepted else REFUSAL,
                                'candidate_source_urls':urls if accepted else [],
                                'refusal_heuristic':is_refusal(answer) if accepted else True,
                                'generated_urls':source_urls(answer) if accepted else [],
                                'seconds':None, 'output_tokens':None})
        print(f"BENCHMARK {case['id']} {method}: {answer}", flush=True)
    predictions.append({**case, 'method':'extractive_bm25',
                        'answer_body':hits[0]['record']['answer'] if accepted else REFUSAL,
                        'retrieved_ids':[h['id'] for h in hits],
                        'candidate_source_urls':[hits[0]['record']['source_url']] if accepted else [],
                        'top1_coverage':hits[0]['coverage'], 'gate_accepted':accepted,
                        'refusal_heuristic':not accepted, 'generated_urls':[], 'seconds':None,
                        'output_tokens':None, 'manual_correct':None, 'manual_refusal':None,'manual_notes':None})
    # Save incrementally so partial results survive a failed session.
    (COMPARISON_DIR/'predictions.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in predictions), encoding='utf-8')

automatic = {}
for method in sorted({r['method'] for r in predictions}):
    rows = [r for r in predictions if r['method']==method]
    positive = [r for r in rows if r['kind']=='answerable']
    negative = [r for r in rows if r['kind']=='unsupported']
    automatic[method] = {'answerable_size':len(positive),'unsupported_size':len(negative),
                         'answerable_refusal_heuristic_count':sum(r['refusal_heuristic'] for r in positive),
                         'unsupported_refusal_heuristic_count':sum(r['refusal_heuristic'] for r in negative),
                         'generated_url_count':sum(len(r['generated_urls']) for r in rows),
                         'reference_in_candidate_sources_count':sum(r['source_url'] in r['candidate_source_urls'] for r in positive)}
manifest = {'gpu':torch.cuda.get_device_name(0),'seed':SEED,'python':platform.python_version(),
            'model_id':MODEL_ID,'model_revision':getattr(model.config,'_commit_hash',None),
            'adapter_source':str(ADAPTER_DIR),'max_new_tokens':160,'do_sample':False,
            'peak_allocated_gpu_bytes':torch.cuda.max_memory_allocated(),
            'versions':{n:importlib.metadata.version(n) for n in ('torch','transformers','peft','bitsandbytes')},
            'benchmark_sha256':hashlib.sha256(json.dumps(benchmark_cases, sort_keys=True).encode()).hexdigest(),
            'calibration_sha256':hashlib.sha256(json.dumps(calibration_cases, sort_keys=True).encode()).hexdigest(),
            'knowledge_base_sha256':hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest(),
            'note':'All 58 facts are available to retrieval. Ten answerable prompts paraphrase the ten fine-tuning holdout facts. Ten unsupported prompts ask for unavailable personal or future details. No independent blinded human assessment. Candidate URLs are retrieval metadata, not proven supporting citations. Gated output reuses rag_base output.'}
for name, obj in [('automatic_metrics.json',automatic),('comparison_manifest.json',manifest),('calibration_result.json',calibration_result)]:
    (COMPARISON_DIR/name).write_text(json.dumps(obj,indent=2),encoding='utf-8')
archive_path = ROOT.parent/'monash_comparison_results.zip'
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in COMPARISON_DIR.iterdir():
        archive.write(path,path.name)
display(FileLink(str(archive_path.relative_to(Path.cwd()))))
print('MONASH_COMPARISON_COMPLETE', flush=True)
